In [1]:
## !pip install captum
## !pip install lime
############################################################
# 0. Imports & config communs
############################################################
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Pour les explications
from captum.attr import IntegratedGradients
from lime.lime_text import LimeTextExplainer
from sklearn.metrics import auc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

BASE_MODEL_NAME = "vinai/bertweet-base"
MAX_LENGTH = 64
BATCH_SIZE = 32
NUM_EPOCHS = 3
LR = 2e-5
LAMBDA_CONTRASTIVE = 1.0
MU_CAUSAL = 1.0
TAU_CONTRASTIVE = 0.1

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
print("BERTweet tokenizer chargé ✅")


C:\Users\mmihoubi\AppData\Local\anaconda3\envs\nrct\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: cpu


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


BERTweet tokenizer chargé ✅


##  1.1 Sentiment140 – sous-échantillon (100k / 20k / 20k)

In [2]:
############################################################
# 1. Chargement Sentiment140
############################################################
SENT140_CSV = "Sentiment140.csv"   # adapte le chemin

def load_sentiment140_subset(
    csv_path=SENT140_CSV,
    n_train=100000,
    n_val=20000,
    n_test=20000,
    random_state=SEED
):
    df = pd.read_csv(csv_path, header=None, encoding="latin-1")
    df.columns = ["target", "ids", "date", "flag", "user", "text"]

    # binaire : 0 négatif, 4 positif
    label_map = {0: 0, 4: 1}
    df = df[df["target"].isin(label_map.keys())].copy()
    df["label"] = df["target"].map(label_map).astype(int)

    print("Distribution labels globale :")
    print(df["label"].value_counts())

    n_total = n_train + n_val + n_test
    n_per_class = n_total // 2  # deux classes, dataset équilibré

    dfs = []
    for lab, group in df.groupby("label"):
        dfs.append(group.sample(n=n_per_class, random_state=random_state))
    df_small = pd.concat(dfs).sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("df_small shape:", df_small.shape)

    # Split train / (val+test)
    df_train, df_temp = train_test_split(
        df_small,
        train_size=n_train,
        random_state=random_state,
        stratify=df_small["label"]
    )
    df_val, df_test = train_test_split(
        df_temp,
        test_size=n_test,
        random_state=random_state,
        stratify=df_temp["label"]
    )

    print("Taille des splits :")
    print("Train:", len(df_train), "Val:", len(df_val), "Test:", len(df_test))

    train_df = df_train[["text", "label"]].reset_index(drop=True)
    val_df   = df_val[["text", "label"]].reset_index(drop=True)
    test_df  = df_test[["text", "label"]].reset_index(drop=True)
    return train_df, val_df, test_df


In [3]:
############################################################
# 2. Chargement TweetEval–Sentiment
############################################################
TWEETEVAL_DIR = "Dataset/sentiments/"  # dossier contenant sentiment_train.csv, etc.

def load_tweeteval_sentiment(data_dir=TWEETEVAL_DIR):
    train_path = os.path.join(data_dir, "sentiment_train.csv")
    val_path   = os.path.join(data_dir, "sentiment_validation.csv")
    test_path  = os.path.join(data_dir, "sentiment_test.csv")

    df_train = pd.read_csv(train_path)
    df_val   = pd.read_csv(val_path)
    df_test  = pd.read_csv(test_path)

    # Adapte si les colonnes ne s'appellent pas 'text' et 'label'
    if "text" not in df_train.columns:
        df_train.columns = ["text", "label"]
        df_val.columns   = ["text", "label"]
        df_test.columns  = ["text", "label"]

    print("TweetEval–Sentiment :")
    print("train shape:", df_train.shape, " label dist:\n", df_train["label"].value_counts())
    print("val   shape:", df_val.shape)
    print("test  shape:", df_test.shape)

    train_df = df_train[["text", "label"]].reset_index(drop=True)
    val_df   = df_val[["text", "label"]].reset_index(drop=True)
    test_df  = df_test[["text", "label"]].reset_index(drop=True)
    return train_df, val_df, test_df


In [4]:
# 2. Dataset PyTorch + bruit

In [5]:
############################################################
# 3. Dataset PyTorch avec bruit optionnel
############################################################
EMOJIS = ["😂", "😍", "😡", "😢", "🙃", "😊"]

def add_noise_to_text(text, char_drop_prob=0.05, char_swap_prob=0.03, emoji_prob=0.05):
    chars = list(text)
    # drop
    new_chars = []
    for c in chars:
        if c != " " and random.random() < char_drop_prob:
            continue
        new_chars.append(c)
    chars = new_chars

    # swap adjacent
    i = 0
    while i < len(chars) - 1:
        if random.random() < char_swap_prob:
            chars[i], chars[i + 1] = chars[i + 1], chars[i]
            i += 2
        else:
            i += 1

    noisy = "".join(chars)

    if random.random() < emoji_prob:
        noisy += " " + random.choice(EMOJIS)

    return noisy


class TweetDataset(Dataset):
    """
    - add_noise=False : clean seulement
    - add_noise=True  : renvoie aussi une vue noisy (pour NRCT et BERTweet+NoiseAug)
    """
    def __init__(self, df, tokenizer, max_length=64, add_noise=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.add_noise = add_noise

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text"])
        label = int(row["label"])

        if self.add_noise:
            noisy_text = add_noise_to_text(text)
        else:
            noisy_text = text

        enc_clean = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        enc_noisy = self.tokenizer(
            noisy_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc_clean["input_ids"].squeeze(0),
            "attention_mask": enc_clean["attention_mask"].squeeze(0),
            "input_ids_noisy": enc_noisy["input_ids"].squeeze(0),
            "attention_mask_noisy": enc_noisy["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


In [6]:
############################################################
# 4. Modèle BERTweet simple (baseline & NoiseAug)
############################################################
class BertweetClassifier(nn.Module):
    def __init__(self, base_model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name, output_attentions=True)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, output_attentions=False):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=output_attentions
        )
        last_hidden = outputs.last_hidden_state  # (B, L, H)
        cls = last_hidden[:, 0, :]               # [CLS]
        logits = self.classifier(cls)

        if output_attentions:
            return logits, outputs.attentions    # liste [num_layers] de (B, n_heads, L, L)
        else:
            return logits


In [7]:
## 3.2 NRCT complet (comme ta version)

In [8]:
############################################################
# 5. NRCT Model (contrastive + causal)
############################################################
class NRCTModel(nn.Module):
    def __init__(self, base_model_name, num_labels=2, proj_dim=128, tau_causal=1.0):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Linear(hidden_size, num_labels)
        self.projection = nn.Linear(hidden_size, proj_dim)

        self.causal_scorer = nn.Linear(hidden_size, 1)
        self.tau_causal = tau_causal

    def encode(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        last_hidden = outputs.last_hidden_state              # (B, L, H)
        h_cls = last_hidden[:, 0, :]                         # (B, H)
        H_tok = last_hidden[:, 1:, :]                        # (B, L-1, H)
        tok_mask = attention_mask[:, 1:]                     # (B, L-1)
        return h_cls, H_tok, tok_mask

    def forward(
        self,
        input_ids,
        attention_mask,
        input_ids_noisy=None,
        attention_mask_noisy=None
    ):
        # clean
        h_cls, H_tok, tok_mask = self.encode(input_ids, attention_mask)
        logits = self.classifier(h_cls)

        # projection
        u = F.normalize(self.projection(h_cls), p=2, dim=-1)

        # causal attention
        scores = self.causal_scorer(H_tok).squeeze(-1)   # (B, T)
        scores = scores.masked_fill(tok_mask == 0, -1e9)
        alpha = torch.softmax(scores / self.tau_causal, dim=-1)  # (B, T)

        # pooled rep (rationales)
        h_rationale = torch.bmm(alpha.unsqueeze(1), H_tok).squeeze(1)
        logits_rationale = self.classifier(h_rationale)

        # noisy branch pour contraste
        u_noisy = None
        if input_ids_noisy is not None and attention_mask_noisy is not None:
            h_cls_noisy, _, _ = self.encode(input_ids_noisy, attention_mask_noisy)
            u_noisy = F.normalize(self.projection(h_cls_noisy), p=2, dim=-1)

        return {
            "logits": logits,
            "u": u,
            "u_noisy": u_noisy,
            "alpha": alpha,
            "logits_rationale": logits_rationale
        }


In [9]:
## 4. Pertes NRCT (classification + contraste + causal)

In [10]:
############################################################
# 6. Pertes pour NRCT
############################################################
def contrastive_info_nce(u, u_pos, temperature=0.1):
    if u_pos is None:
        return torch.tensor(0.0, device=u.device)
    batch_size = u.size(0)
    logits = torch.matmul(u, u_pos.t()) / temperature   # (B, B)
    targets = torch.arange(batch_size, device=u.device)
    return F.cross_entropy(logits, targets)

def causal_loss(logits_full, logits_rationale, alpha, sparsity_weight=0.1):
    with torch.no_grad():
        p = F.softmax(logits_full, dim=-1)
    log_q = F.log_softmax(logits_rationale, dim=-1)
    kl = F.kl_div(log_q, p, reduction="batchmean")
    sparsity = alpha.mean()
    return kl + sparsity_weight * sparsity

def compute_nrct_loss(outputs, labels,
                      lambda_contrastive=1.0,
                      mu_causal=1.0,
                      temperature=0.1):
    logits = outputs["logits"]
    logits_rationale = outputs["logits_rationale"]
    u = outputs["u"]
    u_pos = outputs["u_noisy"]
    alpha = outputs["alpha"]

    loss_cls = F.cross_entropy(logits, labels)
    loss_contr = contrastive_info_nce(u, u_pos, temperature=temperature)
    loss_caus = causal_loss(logits, logits_rationale, alpha, sparsity_weight=0.1)

    loss = loss_cls + lambda_contrastive * loss_contr + mu_causal * loss_caus
    return loss, {
        "total": loss.item(),
        "cls": loss_cls.item(),
        "contrastive": loss_contr.item(),
        "causal": loss_caus.item()
    }


In [11]:
############################################################
# 7. Fonctions d'entraînement et d'évaluation
############################################################
def train_epoch_bert(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = F.cross_entropy(logits, labels)

        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)


def eval_bert(model, loader):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=-1)

            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return acc, f1


def train_epoch_nrct(model, loader, optimizer, scheduler,
                     lambda_contrastive=1.0, mu_causal=1.0,
                     temperature=0.1):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        input_ids_noisy = batch["input_ids_noisy"].to(DEVICE)
        attention_mask_noisy = batch["attention_mask_noisy"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            input_ids_noisy=input_ids_noisy,
            attention_mask_noisy=attention_mask_noisy
        )

        loss, _ = compute_nrct_loss(
            outputs, labels,
            lambda_contrastive=lambda_contrastive,
            mu_causal=mu_causal,
            temperature=temperature
        )
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)


def eval_nrct(model, loader):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                input_ids_noisy=None,
                attention_mask_noisy=None
            )
            logits = outputs["logits"]
            preds = torch.argmax(logits, dim=-1)
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return acc, f1


In [12]:
## 6. Wrapper pour entraîner les 4 modèles sur un dataset donné

In [13]:
############################################################
# 8. Entraînement des 4 modèles sur (train_df, val_df, test_df)
############################################################
def train_all_models_for_dataset(train_df, val_df, test_df, dataset_name="sent140"):
    num_labels = train_df["label"].nunique()
    print(f"\n=== DATASET {dataset_name} | num_labels={num_labels} ===")

    # DataLoaders
    train_clean = TweetDataset(train_df, tokenizer, MAX_LENGTH, add_noise=False)
    train_noisy = TweetDataset(train_df, tokenizer, MAX_LENGTH, add_noise=True)
    val_clean   = TweetDataset(val_df, tokenizer, MAX_LENGTH, add_noise=False)
    test_clean  = TweetDataset(test_df, tokenizer, MAX_LENGTH, add_noise=False)

    train_loader_clean = DataLoader(train_clean, batch_size=BATCH_SIZE, shuffle=True)
    train_loader_noisy = DataLoader(train_noisy, batch_size=BATCH_SIZE, shuffle=True)
    val_loader         = DataLoader(val_clean, batch_size=BATCH_SIZE, shuffle=False)
    test_loader        = DataLoader(test_clean, batch_size=BATCH_SIZE, shuffle=False)

    models = {}

    # ---------- 1) BERTweet baseline ----------
    print("\n=== Entraînement : BERTweet baseline ===")
    model_b = BertweetClassifier(BASE_MODEL_NAME, num_labels=num_labels).to(DEVICE)
    optimizer = torch.optim.AdamW(model_b.parameters(), lr=LR)
    num_steps = NUM_EPOCHS * len(train_loader_clean)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_steps),
        num_training_steps=num_steps
    )
    best_val_f1 = -1
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_epoch_bert(model_b, train_loader_clean, optimizer, scheduler)
        val_acc, val_f1 = eval_bert(model_b, val_loader)
        print(f"[Epoch {epoch}/{NUM_EPOCHS}] Train loss={train_loss:.4f} - Val acc={val_acc:.4f}, Val F1={val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model_b.state_dict().items()}
    model_b.load_state_dict(best_state)
    model_b.to(DEVICE)
    test_acc, test_f1 = eval_bert(model_b, test_loader)
    print(f"[BERTweet baseline] TEST - acc={test_acc:.4f}, F1={test_f1:.4f}")
    models["bert_base"] = model_b

    # ---------- 2) BERTweet + NoiseAug ----------
    print("\n=== Entraînement : BERTweet + NoiseAug ===")
    model_bn = BertweetClassifier(BASE_MODEL_NAME, num_labels=num_labels).to(DEVICE)
    optimizer = torch.optim.AdamW(model_bn.parameters(), lr=LR)
    num_steps = NUM_EPOCHS * len(train_loader_noisy)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_steps),
        num_training_steps=num_steps
    )
    best_val_f1 = -1
    for epoch in range(1, NUM_EPOCHS + 1):
        # IMPORTANT : on utilise la vue noisy comme entrée
        model_bn.train()
        total_loss = 0.0
        for batch in train_loader_noisy:
            optimizer.zero_grad()
            input_ids = batch["input_ids_noisy"].to(DEVICE)
            attention_mask = batch["attention_mask_noisy"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = model_bn(input_ids=input_ids, attention_mask=attention_mask)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item() * len(labels)
        train_loss = total_loss / len(train_loader_noisy.dataset)
        val_acc, val_f1 = eval_bert(model_bn, val_loader)
        print(f"[Epoch {epoch}/{NUM_EPOCHS}] Train loss={train_loss:.4f} - Val acc={val_acc:.4f}, Val F1={val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model_bn.state_dict().items()}
    model_bn.load_state_dict(best_state)
    model_bn.to(DEVICE)
    test_acc, test_f1 = eval_bert(model_bn, test_loader)
    print(f"[BERTweet+NoiseAug] TEST - acc={test_acc:.4f}, F1={test_f1:.4f}")
    models["bert_noiseaug"] = model_bn

    # ---------- 3) NRCT sans contraste ----------
    print("\n=== Entraînement : NRCT sans contraste (lambda=0) ===")
    model_nc = NRCTModel(BASE_MODEL_NAME, num_labels=num_labels, proj_dim=128, tau_causal=1.0).to(DEVICE)
    optimizer = torch.optim.AdamW(model_nc.parameters(), lr=LR)
    num_steps = NUM_EPOCHS * len(train_loader_noisy)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_steps),
        num_training_steps=num_steps
    )
    best_val_f1 = -1
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_epoch_nrct(
            model_nc, train_loader_noisy, optimizer, scheduler,
            lambda_contrastive=0.0,   # pas de contraste
            mu_causal=MU_CAUSAL,
            temperature=TAU_CONTRASTIVE
        )
        val_acc, val_f1 = eval_nrct(model_nc, val_loader)
        print(f"[Epoch {epoch}/{NUM_EPOCHS}] Train loss={train_loss:.4f} - Val acc={val_acc:.4f}, Val F1={val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model_nc.state_dict().items()}
    model_nc.load_state_dict(best_state)
    model_nc.to(DEVICE)
    test_acc, test_f1 = eval_nrct(model_nc, test_loader)
    print(f"[NRCT w/o contrastive] TEST - acc={test_acc:.4f}, F1={test_f1:.4f}")
    models["nrct_no_contrast"] = model_nc

    # ---------- 4) NRCT full ----------
    print("\n=== Entraînement : NRCT full (contrastive + causal) ===")
    model_nf = NRCTModel(BASE_MODEL_NAME, num_labels=num_labels, proj_dim=128, tau_causal=1.0).to(DEVICE)
    optimizer = torch.optim.AdamW(model_nf.parameters(), lr=LR)
    num_steps = NUM_EPOCHS * len(train_loader_noisy)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * num_steps),
        num_training_steps=num_steps
    )
    best_val_f1 = -1
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_epoch_nrct(
            model_nf, train_loader_noisy, optimizer, scheduler,
            lambda_contrastive=LAMBDA_CONTRASTIVE,
            mu_causal=MU_CAUSAL,
            temperature=TAU_CONTRASTIVE
        )
        val_acc, val_f1 = eval_nrct(model_nf, val_loader)
        print(f"[Epoch {epoch}/{NUM_EPOCHS}] Train loss={train_loss:.4f} - Val acc={val_acc:.4f}, Val F1={val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model_nf.state_dict().items()}
    model_nf.load_state_dict(best_state)
    model_nf.to(DEVICE)
    test_acc, test_f1 = eval_nrct(model_nf, test_loader)
    print(f"[NRCT full] TEST - acc={test_acc:.4f}, F1={test_f1:.4f}")
    models["nrct_full"] = model_nf

    return models, (train_clean, val_clean, test_clean)


In [14]:
############################################################
# 9. Robustesse au bruit
############################################################
def evaluate_noise_bert(model, df_split, noise_settings,
                        tokenizer, max_length=64, batch_size=32):
    results = []
    for (drop_p, swap_p, emoji_p) in noise_settings:
        class NoisyDS(Dataset):
            def __init__(self, df):
                self.df = df.reset_index(drop=True)
            def __len__(self):
                return len(self.df)
            def __getitem__(self, idx):
                row = self.df.iloc[idx]
                text = add_noise_to_text(
                    str(row["text"]),
                    char_drop_prob=drop_p,
                    char_swap_prob=swap_p,
                    emoji_prob=emoji_p
                )
                label = int(row["label"])
                enc = tokenizer(
                    text,
                    padding="max_length",
                    truncation=True,
                    max_length=max_length,
                    return_tensors="pt"
                )
                return {
                    "input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "labels": torch.tensor(label, dtype=torch.long)
                }

        ds = NoisyDS(df_split)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
        acc, f1 = eval_bert(model, loader)
        results.append((drop_p, acc, f1))
        print(f"Noise (drop={drop_p}, swap={swap_p}, emoji={emoji_p}) => acc={acc:.4f}, F1={f1:.4f}")
    return results


def evaluate_noise_nrct(model, df_split, noise_settings,
                        tokenizer, max_length=64, batch_size=32):
    results = []
    for (drop_p, swap_p, emoji_p) in noise_settings:
        class NoisyDS(Dataset):
            def __init__(self, df):
                self.df = df.reset_index(drop=True)
            def __len__(self):
                return len(self.df)
            def __getitem__(self, idx):
                row = self.df.iloc[idx]
                text = add_noise_to_text(
                    str(row["text"]),
                    char_drop_prob=drop_p,
                    char_swap_prob=swap_p,
                    emoji_prob=emoji_p
                )
                label = int(row["label"])
                enc = tokenizer(
                    text,
                    padding="max_length",
                    truncation=True,
                    max_length=max_length,
                    return_tensors="pt"
                )
                return {
                    "input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "labels": torch.tensor(label, dtype=torch.long)
                }

        ds = NoisyDS(df_split)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
        acc, f1 = eval_nrct(model, loader)
        results.append((drop_p, acc, f1))
        print(f"Noise (drop={drop_p}, swap={swap_p}, emoji={emoji_p}) => acc={acc:.4f}, F1={f1:.4f}")
    return results


def run_robustness_suite(models, test_df, dataset_name="sent140"):
    print(f"\n=== Robustesse au bruit sur le TEST pour {dataset_name} ===")
    noise_settings = [
        (0.00, 0.00, 0.00),
        (0.03, 0.02, 0.02),
        (0.05, 0.03, 0.05),
        (0.10, 0.05, 0.05),
    ]
    rob = {}
    print("\n--- BERTweet baseline ---")
    rob["bert_base"] = evaluate_noise_bert(models["bert_base"], test_df, noise_settings, tokenizer, MAX_LENGTH, BATCH_SIZE)
    print("\n--- BERTweet+NoiseAug ---")
    rob["bert_noiseaug"] = evaluate_noise_bert(models["bert_noiseaug"], test_df, noise_settings, tokenizer, MAX_LENGTH, BATCH_SIZE)
    print("\n--- NRCT-no-contrast ---")
    rob["nrct_no_contrast"] = evaluate_noise_nrct(models["nrct_no_contrast"], test_df, noise_settings, tokenizer, MAX_LENGTH, BATCH_SIZE)
    print("\n--- NRCT-full ---")
    rob["nrct_full"] = evaluate_noise_nrct(models["nrct_full"], test_df, noise_settings, tokenizer, MAX_LENGTH, BATCH_SIZE)
    return rob


In [15]:
############################################################
# 10. Explications : rationales + AUC (deletion / insertion)
############################################################
def compute_auc_deletion_insertion(probs_seq):
    """
    probs_seq : liste des probas (entre 0 et 1) à chaque étape de suppression / insertion.
    On suppose probas normalisées sur [0,1].
    """
    xs = np.linspace(0, 1, num=len(probs_seq))
    ys = np.array(probs_seq)
    return auc(xs, ys)


In [16]:
def explain_nrct_on_batch(model, batch, top_ratio=0.3):
    """
    Retourne :
      - lengths : nb tokens sélectionnés
      - deletion_probs, insertion_probs : listes de séquences de proba (une par exemple)
    """
    model.eval()
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        input_ids_noisy=None,
                        attention_mask_noisy=None)
        logits = outputs["logits"]
        alpha = outputs["alpha"]          # (B, T)
        probs = F.softmax(logits, dim=-1) # (B, C)

    lengths = []
    del_probas = []
    ins_probas = []

    for i in range(input_ids.size(0)):
        label = labels[i].item()
        token_ids = input_ids[i].clone()
        mask = attention_mask[i].clone()
        L = mask.sum().item() - 1  # nb tokens non [CLS] (on a alpha pour ces tokens)

        k = max(1, int(top_ratio * L))
        scores = alpha[i][:L]   # (T_effective,)
        top_idx = torch.argsort(scores, descending=True)[:k]  # indices dans [1..L]
        # on décalera de +1 pour l'indice dans la séquence complète (car 0 = CLS)
        true_indices = (top_idx + 1).cpu().tolist()
        lengths.append(len(true_indices))

        # ----- Deletion -----
        ids_del = token_ids.clone()
        mask_del = mask.clone()
        prob_seq_del = []
        for j in range(len(true_indices) + 1):
            # proba actuelle
            with torch.no_grad():
                out = model(input_ids=ids_del.unsqueeze(0),
                            attention_mask=mask_del.unsqueeze(0),
                            input_ids_noisy=None,
                            attention_mask_noisy=None)
                logit_j = out["logits"][0]
                p = F.softmax(logit_j, dim=-1)[label].item()
            prob_seq_del.append(p)

            if j < len(true_indices):
                idx_tok = true_indices[j]
                ids_del[idx_tok] = tokenizer.mask_token_id
        del_probas.append(prob_seq_del)

        # ----- Insertion -----
        # on part de séquence où les top tokens sont masqués
        ids_ins = token_ids.clone()
        mask_ins = mask.clone()
        for idx_tok in true_indices:
            ids_ins[idx_tok] = tokenizer.mask_token_id

        prob_seq_ins = []
        for j in range(len(true_indices) + 1):
            with torch.no_grad():
                out = model(input_ids=ids_ins.unsqueeze(0),
                            attention_mask=mask_ins.unsqueeze(0),
                            input_ids_noisy=None,
                            attention_mask_noisy=None)
                logit_j = out["logits"][0]
                p = F.softmax(logit_j, dim=-1)[label].item()
            prob_seq_ins.append(p)
            if j < len(true_indices):
                idx_tok = true_indices[j]
                ids_ins[idx_tok] = token_ids[idx_tok]
        ins_probas.append(prob_seq_ins)

    return lengths, del_probas, ins_probas


In [17]:
def explain_attention_bert_on_batch(model, batch, top_ratio=0.3):
    model.eval()
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)

    logits, attentions = model(input_ids=input_ids,
                               attention_mask=attention_mask,
                               output_attentions=True)
    probs = F.softmax(logits, dim=-1)

    last_att = attentions[-1]   # (B, n_heads, L, L)
    att_cls = last_att.mean(dim=1)[:, 0, :]  # (B, L) attention de CLS vers chaque token

    lengths, del_probas, ins_probas = [], [], []

    for i in range(input_ids.size(0)):
        label = labels[i].item()
        token_ids = input_ids[i].clone()
        mask = attention_mask[i].clone()
        L = mask.sum().item()
        scores = att_cls[i, :L]
        k = max(1, int(top_ratio * (L - 1)))

        top_idx = torch.argsort(scores[1:], descending=True)[:k]  # on ignore CLS
        true_indices = (top_idx + 1).cpu().tolist()
        lengths.append(len(true_indices))

        # deletion
        ids_del = token_ids.clone()
        mask_del = mask.clone()
        prob_seq_del = []
        for j in range(len(true_indices) + 1):
            with torch.no_grad():
                logit_j = model(
                    input_ids=ids_del.unsqueeze(0),
                    attention_mask=mask_del.unsqueeze(0)
                )
                p = F.softmax(logit_j, dim=-1)[0, label].item()
            prob_seq_del.append(p)
            if j < len(true_indices):
                idx_tok = true_indices[j]
                ids_del[idx_tok] = tokenizer.mask_token_id
        del_probas.append(prob_seq_del)

        # insertion
        ids_ins = token_ids.clone()
        mask_ins = mask.clone()
        for idx_tok in true_indices:
            ids_ins[idx_tok] = tokenizer.mask_token_id

        prob_seq_ins = []
        for j in range(len(true_indices) + 1):
            with torch.no_grad():
                logit_j = model(
                    input_ids=ids_ins.unsqueeze(0),
                    attention_mask=mask_ins.unsqueeze(0)
                )
                p = F.softmax(logit_j, dim=-1)[0, label].item()
            prob_seq_ins.append(p)
            if j < len(true_indices):
                idx_tok = true_indices[j]
                ids_ins[idx_tok] = token_ids[idx_tok]
        ins_probas.append(prob_seq_ins)

    return lengths, del_probas, ins_probas


In [18]:
def explain_attention_bert_on_batch(model, batch, top_ratio=0.3):
    model.eval()
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)

    logits, attentions = model(input_ids=input_ids,
                               attention_mask=attention_mask,
                               output_attentions=True)
    probs = F.softmax(logits, dim=-1)

    last_att = attentions[-1]   # (B, n_heads, L, L)
    att_cls = last_att.mean(dim=1)[:, 0, :]  # (B, L) attention de CLS vers chaque token

    lengths, del_probas, ins_probas = [], [], []

    for i in range(input_ids.size(0)):
        label = labels[i].item()
        token_ids = input_ids[i].clone()
        mask = attention_mask[i].clone()
        L = mask.sum().item()
        scores = att_cls[i, :L]
        k = max(1, int(top_ratio * (L - 1)))

        top_idx = torch.argsort(scores[1:], descending=True)[:k]  # on ignore CLS
        true_indices = (top_idx + 1).cpu().tolist()
        lengths.append(len(true_indices))

        # deletion
        ids_del = token_ids.clone()
        mask_del = mask.clone()
        prob_seq_del = []
        for j in range(len(true_indices) + 1):
            with torch.no_grad():
                logit_j = model(
                    input_ids=ids_del.unsqueeze(0),
                    attention_mask=mask_del.unsqueeze(0)
                )
                p = F.softmax(logit_j, dim=-1)[0, label].item()
            prob_seq_del.append(p)
            if j < len(true_indices):
                idx_tok = true_indices[j]
                ids_del[idx_tok] = tokenizer.mask_token_id
        del_probas.append(prob_seq_del)

        # insertion
        ids_ins = token_ids.clone()
        mask_ins = mask.clone()
        for idx_tok in true_indices:
            ids_ins[idx_tok] = tokenizer.mask_token_id

        prob_seq_ins = []
        for j in range(len(true_indices) + 1):
            with torch.no_grad():
                logit_j = model(
                    input_ids=ids_ins.unsqueeze(0),
                    attention_mask=mask_ins.unsqueeze(0)
                )
                p = F.softmax(logit_j, dim=-1)[0, label].item()
            prob_seq_ins.append(p)
            if j < len(true_indices):
                idx_tok = true_indices[j]
                ids_ins[idx_tok] = token_ids[idx_tok]
        ins_probas.append(prob_seq_ins)

    return lengths, del_probas, ins_probas


In [19]:
def explain_ig_bert_on_batch(model, batch, top_ratio=0.3, steps=16):
    """
    IG approximatif sur les embeddings de BERTweet.
    On renvoie des rationales + courbes deletion/insertion comme pour les autres.
    """
    model.eval()
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)

    embedding_layer = model.encoder.embeddings.word_embeddings

    lengths, del_probas, ins_probas = [], [], []

    for i in range(input_ids.size(0)):
        ids = input_ids[i].unsqueeze(0)          # (1, L)
        mask = attention_mask[i].unsqueeze(0)
        label = labels[i].item()

        # embeddings de base (baseline = 0)
        emb = embedding_layer(ids)

        total_grad = torch.zeros_like(emb)
        for alpha in np.linspace(0, 1, steps):
            emb_scaled = emb * alpha
            emb_scaled.requires_grad_(True)
            outputs = model.encoder(inputs_embeds=emb_scaled, attention_mask=mask)
            cls = outputs.last_hidden_state[:, 0, :]
            logits = model.classifier(cls)
            prob = F.softmax(logits, dim=-1)[0, label]
            prob.backward(retain_graph=True)
            total_grad += emb_scaled.grad.detach()
            model.zero_grad()

        attributions = (total_grad * emb).sum(dim=-1).squeeze(0)    # (L,)
        L = mask.sum().item()
        scores = attributions[:L]
        k = max(1, int(top_ratio * (L - 1)))
        top_idx = torch.argsort(scores[1:], descending=True)[:k]
        true_indices = (top_idx + 1).cpu().tolist()
        lengths.append(len(true_indices))

        token_ids = ids.squeeze(0).clone()
        mask_vec = mask.squeeze(0).clone()

        # deletion
        ids_del = token_ids.clone()
        prob_seq_del = []
        for j in range(len(true_indices) + 1):
            with torch.no_grad():
                logit_j = model(input_ids=ids_del.unsqueeze(0),
                                attention_mask=mask_vec.unsqueeze(0))
                p = F.softmax(logit_j, dim=-1)[0, label].item()
            prob_seq_del.append(p)
            if j < len(true_indices):
                idx_tok = true_indices[j]
                ids_del[idx_tok] = tokenizer.mask_token_id
        del_probas.append(prob_seq_del)

        # insertion
        ids_ins = token_ids.clone()
        for idx_tok in true_indices:
            ids_ins[idx_tok] = tokenizer.mask_token_id
        prob_seq_ins = []
        for j in range(len(true_indices) + 1):
            with torch.no_grad():
                logit_j = model(input_ids=ids_ins.unsqueeze(0),
                                attention_mask=mask_vec.unsqueeze(0))
                p = F.softmax(logit_j, dim=-1)[0, label].item()
            prob_seq_ins.append(p)
            if j < len(true_indices):
                idx_tok = true_indices[j]
                ids_ins[idx_tok] = token_ids[idx_tok]
        ins_probas.append(prob_seq_ins)

    return lengths, del_probas, ins_probas


In [20]:
def predict_proba_bert_raw_text(texts, model, tokenizer, max_length=64):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for t in texts:
            enc = tokenizer(
                t,
                padding="max_length",
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )
            input_ids = enc["input_ids"].to(DEVICE)
            attn = enc["attention_mask"].to(DEVICE)
            logits = model(input_ids=input_ids, attention_mask=attn)
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs[0])
    return np.vstack(all_probs)

def explain_lime_bert_on_batch(model, df_batch, top_ratio=0.3, num_features=10):
    """
    df_batch : DataFrame avec colonnes text, label (quelques dizaines d'exemples).
    On renvoie longueur moyenne + deletion / insertion approximatives
    (on ne refait pas la boucle token par token, car LIME travaille sur les mots).
    """
    explainer = LimeTextExplainer(class_names=[str(i) for i in sorted(df_batch["label"].unique())])

    lengths = []
    del_probas = []
    ins_probas = []

    for _, row in df_batch.iterrows():
        text = row["text"]
        label = int(row["label"])

        exp = explainer.explain_instance(
            text,
            lambda xs: predict_proba_bert_raw_text(xs, model, tokenizer, MAX_LENGTH),
            labels=[label],
            num_features=num_features
        )
        # mots les plus importants pour ce label
        weights = exp.as_map()[label]   # liste (index_mot, poids)
        weights_sorted = sorted(weights, key=lambda x: abs(x[1]), reverse=True)
        k = max(1, int(top_ratio * len(weights_sorted)))
        top_indices = [idx for idx, w in weights_sorted[:k]]
        lengths.append(len(top_indices))

        # pour rester simple, on approxime deletion/insertion en masquant les mots
        tokens = text.split()
        # deletion
        prob_seq_del = []
        current_tokens = tokens.copy()
        for j in range(len(top_indices) + 1):
            sent = " ".join(current_tokens)
            p = predict_proba_bert_raw_text([sent], model, tokenizer, MAX_LENGTH)[0, label]
            prob_seq_del.append(p)
            if j < len(top_indices):
                idx = top_indices[j]
                if idx < len(current_tokens):
                    current_tokens[idx] = ""   # suppr simple
        del_probas.append(prob_seq_del)

        # insertion
        tokens_masked = tokens.copy()
        for idx in top_indices:
            if idx < len(tokens_masked):
                tokens_masked[idx] = ""
        prob_seq_ins = []
        for j in range(len(top_indices) + 1):
            sent = " ".join(tokens_masked)
            p = predict_proba_bert_raw_text([sent], model, tokenizer, MAX_LENGTH)[0, label]
            prob_seq_ins.append(p)
            if j < len(top_indices):
                idx = top_indices[j]
                if idx < len(tokens_masked):
                    tokens_masked[idx] = tokens[idx]
        ins_probas.append(prob_seq_ins)

    return lengths, del_probas, ins_probas


In [21]:
def explanation_metrics_from_lengths_and_curves(lengths, del_probas, ins_probas):
    avg_len = float(np.mean(lengths))
    del_aucs = [compute_auc_deletion_insertion(seq) for seq in del_probas]
    ins_aucs = [compute_auc_deletion_insertion(seq) for seq in ins_probas]
    return avg_len, float(np.mean(del_aucs)), float(np.mean(ins_aucs))


def run_explanations_suite(models, test_df, dataset_name="sent140", n_examples=500):
    """
    Calcule les métriques d'explication pour :
      - NRCT causal attention
      - BERT attention
      - BERT + IG
      - BERT + LIME
    sur un sous-ensemble de test.
    """
    print(f"\n=== Explications sur {dataset_name} (n={n_examples}) ===")
    df_sub = test_df.sample(n=min(n_examples, len(test_df)), random_state=SEED).reset_index(drop=True)
    ds_sub = TweetDataset(df_sub, tokenizer, MAX_LENGTH, add_noise=False)
    loader_sub = DataLoader(ds_sub, batch_size=16, shuffle=False)

    # ----- NRCT causal attention -----
    print("\n--- NRCT causal attention ---")
    lengths_all, del_all, ins_all = [], [], []
    for batch in loader_sub:
        l, d, ins = explain_nrct_on_batch(models["nrct_full"], batch, top_ratio=0.3)
        lengths_all.extend(l)
        del_all.extend(d)
        ins_all.extend(ins)
    nrct_len, nrct_del_auc, nrct_ins_auc = explanation_metrics_from_lengths_and_curves(
        lengths_all, del_all, ins_all
    )
    print("NRCT - avg_len:", nrct_len, " delAUC:", nrct_del_auc, " insAUC:", nrct_ins_auc)

    # ----- BERT attention -----
    print("\n--- BERT attention ---")
    lengths_all, del_all, ins_all = [], [], []
    for batch in loader_sub:
        l, d, ins = explain_attention_bert_on_batch(models["bert_base"], batch, top_ratio=0.3)
        lengths_all.extend(l)
        del_all.extend(d)
        ins_all.extend(ins)
    att_len, att_del_auc, att_ins_auc = explanation_metrics_from_lengths_and_curves(
        lengths_all, del_all, ins_all
    )
    print("BERT-attention - avg_len:", att_len, " delAUC:", att_del_auc, " insAUC:", att_ins_auc)

    # ----- BERT + IG -----
    print("\n--- BERT + IG ---")
    lengths_all, del_all, ins_all = [], [], []
    for batch in loader_sub:
        l, d, ins = explain_ig_bert_on_batch(models["bert_base"], batch, top_ratio=0.3, steps=16)
        lengths_all.extend(l)
        del_all.extend(d)
        ins_all.extend(ins)
    ig_len, ig_del_auc, ig_ins_auc = explanation_metrics_from_lengths_and_curves(
        lengths_all, del_all, ins_all
    )
    print("BERT-IG - avg_len:", ig_len, " delAUC:", ig_del_auc, " insAUC:", ig_ins_auc)

    # ----- BERT + LIME (sur un sous-sous-ensemble) -----
    print("\n--- BERT + LIME ---")
    df_lime = df_sub.iloc[: min(200, len(df_sub))].copy()  # LIME est plus cher
    lengths, del_p, ins_p = explain_lime_bert_on_batch(models["bert_base"], df_lime, top_ratio=0.3)
    lime_len, lime_del_auc, lime_ins_auc = explanation_metrics_from_lengths_and_curves(
        lengths, del_p, ins_p
    )
    print("BERT-LIME - avg_len:", lime_len, " delAUC:", lime_del_auc, " insAUC:", lime_ins_auc)

    results = {
        "nrct":  (nrct_len, nrct_del_auc, nrct_ins_auc),
        "att":   (att_len, att_del_auc, att_ins_auc),
        "ig":    (ig_len, ig_del_auc, ig_ins_auc),
        "lime":  (lime_len, lime_del_auc, lime_ins_auc),
    }
    return results


In [ ]:
# === TweetEval–Sentiment ===
train_te, val_te, test_te = load_tweeteval_sentiment(TWEETEVAL_DIR)

models_te, _ = train_all_models_for_dataset(train_te, val_te, test_te, dataset_name="TweetEval-Sentiment")
robust_te = run_robustness_suite(models_te, test_te, dataset_name="TweetEval-Sentiment")
expl_te = run_explanations_suite(models_te, test_te, dataset_name="TweetEval-Sentiment", n_examples=500)


TweetEval–Sentiment :
train shape: (45615, 2)  label dist:
 label
1    20673
2    17849
0     7093
Name: count, dtype: int64
val   shape: (2000, 2)
test  shape: (12284, 2)

=== DATASET TweetEval-Sentiment | num_labels=3 ===

=== Entraînement : BERTweet baseline ===


In [ ]:
# === TweetEval–Sentiment ===
train_te, val_te, test_te = load_tweeteval_sentiment(TWEETEVAL_DIR)

models_te, _ = train_all_models_for_dataset(train_te, val_te, test_te, dataset_name="TweetEval-Sentiment")
robust_te = run_robustness_suite(models_te, test_te, dataset_name="TweetEval-Sentiment")
expl_te = run_explanations_suite(models_te, test_te, dataset_name="TweetEval-Sentiment", n_examples=500)
